In [3]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import os
import networkx as nx 
from collections import defaultdict
from dateutil.parser import parse
from networkx.algorithms import cluster


In [4]:
def build_graph(services_data: dict) -> nx.DiGraph:
    g = nx.DiGraph()
    for svc in services_data['services']:
        g.add_node(svc['name'], **{k: v for k, v in svc.items() if k != 'name'})
        
    for store in services_data.get('stores', []): 
        g.add_node(store['name'], **{k: v for k, v in store.items() if k != 'name'})
        
    for edge in services_data.get('edges', []): 
        g.add_edge(edge['from'], edge['to'], type=edge['type'])
        
    return g


In [5]:
def open_file(cluster_path, service_path, alert_path, incident_path):
    with open(cluster_path, 'r', encoding='utf8') as file:
        cluster_summary = json.load(file)

    with open(service_path, 'r',  encoding='utf8') as file:
        services_data = json.load(file)
        
    with open(alert_path, 'r',  encoding='utf8') as file:
        alerts = [json.loads(line) for line in file if line.strip()]

    with open(incident_path, 'r',  encoding='utf8') as file:
        incidents = json.load(file)['incidents']

    return cluster_summary, services_data, alerts, incidents

In [6]:
def score_candidate(service, alert, graph):
    subgraph = graph.subgraph(service)

    try:
        pagerank = nx.pagerank(subgraph.reverse(copy=True), alpha=0.85)
    except Exception:
        pagerank = {svc: 0 for svc in service}

    max_pr = max(pagerank.values()) if pagerank else 1.0
    if max_pr == 0: max_pr = 1.0
    norm_pr_scores = {k: v / max_pr for k, v in pagerank.items()}

    earliest_time = float('inf')
    earliest_svc = None

    for a in alert:
        svc = a.get('service') # ĐÃ SỬA: 'service' (số ít) thay vì 'services'
        ts = a.get('ts') or a.get('timestamp') 
        
        if svc in service and ts < earliest_time:
            earliest_time = ts
            earliest_svc = svc

    scores = {} # ĐÃ SỬA: Đổi từ [] thành {}

    for svc in service:
        pagerank_norm = norm_pr_scores.get(svc, 0.0)
        timestamp_score = 1.0 if svc == earliest_svc else 0.0
        scores[svc] = (0.6 * pagerank_norm ) + (0.4 * timestamp_score)

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_scores

In [7]:
def top_incident(service, history_data):
    best_match = None
    max_score = -1
    cluster_set = set(service)

    for past in history_data:
        past_services = set(past.get('services_involved', []))
        
        length = len(cluster_set & past_services)
        score = length * 0.2

        if past.get('root_cause_service') in service:
            score += 0.4

        if score > max_score and score >= 0.2: 
            max_score = score
            best_match = past
            
    return best_match, max_score

In [8]:
def pipeline():
    cluster_summary, services_data, alerts_raw, incidents = open_file(
        cluster_path="../d1/results/cluster_summary.json",
        service_path="dataset/services.json",
        alert_path="dataset/alerts_sample.jsonl",
        incident_path="dataset/incidents_history.json"
    )

    graph = build_graph(services_data)
    results = []

    cluster_list = cluster_summary.get('clusters', cluster_summary) if isinstance(cluster_summary, dict) else cluster_summary

    for cluster in cluster_list:
        cluster_id = cluster.get('cluster_id')
        cluster_services = cluster.get('services', [])

        if not cluster_services:
            continue

        cluster_alerts = [a for a in alerts_raw if a.get('alert_id') in cluster.get('alert_ids', [])]

        candidate = score_candidate(cluster_services, cluster_alerts, graph)
        top_k = [[c[0], round(c[1], 2)] for c in candidate[:3]]

        root_cause = top_k[0][0] if top_k else 'unknown'
        similar_incident, sim_score = top_incident(cluster_services, incidents)

        if similar_incident:
            pred_class = similar_incident.get('root_cause_class', similar_incident.get('class', 'other'))
            raw_actions = similar_incident.get('actions', similar_incident.get('remediation', "Investigate manually"))
            actions = raw_actions.split('. ') if isinstance(raw_actions, str) else raw_actions
            similar_ids = [similar_incident.get('incident_id', similar_incident.get('id', ''))]
            confidence = round(sim_score, 2)
        else:
            pred_class = "other"
            actions = ["Investigate manually"]
            similar_ids = []
            confidence = 0.0

        results.append({
            "cluster_id": cluster_id,
            "graph_top3": top_k,
            "root_cause": root_cause,
            "class": pred_class,
            "confidence": confidence,
            "actions": [a.strip() for a in actions if a.strip()],
            "reasoning": f"Service {root_cause} identified via topology (PageRank) and temporal alerts. Retrieved past incident with similarity score {confidence}.",
            "similar_incidents": similar_ids,
            "method": "graph+retrieval"
        })

    output = {"clusters_analyzed": len(results), "results": results}
    os.makedirs("results", exist_ok=True)
    output_path = "results/rca_output.json"
    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(output, file, indent=2, ensure_ascii=False)

    print(f"{output['clusters_analyzed']} clusters.")

In [9]:
result = pipeline()

5 clusters.
